<a href="https://colab.research.google.com/github/lavaeagle2/scifi-hackathon/blob/main/trials_and_errors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from sklearn.ensemble import IsolationForest
# from sklearn.preprocessing import StandardScaler
# from sklearn.preprocessing import StandardScaler
# from sklearn.ensemble import IsolationForest
# from sklearn.neighbors import LocalOutlierFactor

In [ ]:
df_train = pd.read_csv('train_vitals.csv')
df_test  = pd.read_csv('test_vitals.csv')

In [ ]:
# # sorting
# df_train = df_train.sort_values(['case_id', 'time_sec'])
# df_test  = df_test.sort_values(['case_id', 'time_sec'])


In [ ]:
# #Feature Engineering
# for df in [df_train, df_test]:
#     df['shock_index'] = df['HR'] / (df['MBP'] + 1e-6)
#     df['HR_diff'] = df.groupby('case_id')['HR'].diff().fillna(0)
#     df['SpO2_diff'] = df.groupby('case_id')['SpO2'].diff().fillna(0)
#     df['Temp_diff'] = df.groupby('case_id')['Temp'].diff().fillna(0)
#     df['HR_roll_mean'] = df.groupby('case_id')['HR'].rolling(5).mean().reset_index(0, drop=True)
#     df['HR_roll_std']  = df.groupby('case_id')['HR'].rolling(5).std().reset_index(0, drop=True)

# FEATURES = [
#     'HR','MBP','SpO2','Temp',
#     'shock_index',
#     'HR_diff','SpO2_diff','Temp_diff',
#     'HR_roll_mean','HR_roll_std'
# ]

In [ ]:
# #Handle missing values
# df_train[FEATURES] = df_train[FEATURES].fillna(df_train[FEATURES].median())
# df_test[FEATURES]  = df_test[FEATURES].fillna(df_train[FEATURES].median())

In [ ]:
# #Scaling
# scaler = StandardScaler()

# X_train = scaler.fit_transform(df_train[FEATURES])
# X_test  = scaler.transform(df_test[FEATURES])

In [ ]:
# models = [
#     IsolationForest(n_estimators=200, contamination=0.01, random_state=42, n_jobs=-1),
#     IsolationForest(n_estimators=200, contamination=0.03, random_state=99, n_jobs=-1),
#     IsolationForest(n_estimators=200, contamination=0.05, random_state=7,  n_jobs=-1),
# ]

# scores = []


In [ ]:
# for model in models:
#     model.fit(X_train)
#     s = -model.decision_function(X_test)
#     scores.append(s)

# # Combine scores
# final_scores = np.mean(scores, axis=0)

In [ ]:
# final_scores = (final_scores - final_scores.min()) / (final_scores.max() - final_scores.min())
# final_scores = pd.Series(final_scores).rank(pct=True).values
# final_scores = np.clip(final_scores, 0.01, 0.99)

# #submission
# submission = pd.DataFrame({
#     'case_id': df_test['case_id'],
#     'time_sec': df_test['time_sec'],
#     'anomaly_score': final_scores
# })

# submission.to_csv('my_submission.csv', index=False)

# print(submission.head())




In [ ]:
# """
# =============================================================
# ICU Anomaly Detection — VitalDB  (v2 — tuned for your data)
# Columns: case_id, time_sec, hr, spo2, mbp, temp (sparse)
# Fully unsupervised | Target: ROC-AUC · PR-AUC · F1 · Recall
# =============================================================
# """

# # ── 0. INSTALLS ───────────────────────────────────────────────────────────────
# import subprocess, sys
# def pip(pkg): subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
# pip("pyod")

# import numpy as np
# import pandas as pd
# from sklearn.preprocessing import RobustScaler
# from sklearn.cluster import MiniBatchKMeans
# from sklearn.ensemble import IsolationForest
# from pyod.models.hbos  import HBOS
# from pyod.models.copod import COPOD
# from pyod.models.ecod  import ECOD

# # ── 1. LOAD ───────────────────────────────────────────────────────────────────
# train = pd.read_csv("train_vitals.csv")
# test  = pd.read_csv("test_vitals.csv")

# print("Train:", train.shape, "| Test:", test.shape)
# print("Columns:", train.columns.tolist())
# print(train.head(3))
# print("\nMissing %:\n", (train.isnull().mean()*100).round(1))

# # ── 2. COLUMN SETUP ───────────────────────────────────────────────────────────
# # Standardise names — handles hr/HR/heart_rate etc.
# rename = {}
# for col in train.columns:
#     cl = col.lower().strip()
#     if cl in ["hr","heart_rate","heartrate"]:       rename[col]="HR"
#     elif cl in ["spo2","o2sat","oxygen"]:            rename[col]="SpO2"
#     elif cl in ["mbp","map","mean_bp","mean_abp"]:   rename[col]="MBP"
#     elif cl in ["temp","temperature","bt"]:          rename[col]="Temp"

# train.rename(columns=rename, inplace=True)
# test.rename(columns=rename, inplace=True)

# # Core vitals — only include Temp if it has >20% non-null values
# CORE = ["HR","SpO2","MBP"]
# if "Temp" in train.columns and train["Temp"].notna().mean() > 0.20:
#     CORE.append("Temp")
#     print("✅ Temp included")
# else:
#     print("⚠️  Temp excluded (too sparse — would add noise, not signal)")

# print("Active vitals:", CORE)

# # ── 3. SORT & CLIP ────────────────────────────────────────────────────────────
# for df in [train, test]:
#     df.sort_values(["case_id","time_sec"], inplace=True)
#     df.reset_index(drop=True, inplace=True)

# BOUNDS = {"HR":(20,300), "SpO2":(50,100), "MBP":(20,200), "Temp":(30,43)}
# for df in [train, test]:
#     for col, (lo, hi) in BOUNDS.items():
#         if col in df.columns:
#             df[col] = df[col].clip(lo, hi)

# # ── 4. IMPUTATION ─────────────────────────────────────────────────────────────
# # Per-patient forward+backward fill, then global median fallback
# for df in [train, test]:
#     df[CORE] = df.groupby("case_id")[CORE].transform(
#         lambda g: g.ffill().bfill()
#     )
#     df[CORE] = df[CORE].fillna(df[CORE].median())

# # ── 5. FEATURE ENGINEERING ────────────────────────────────────────────────────
# def make_features(df, core):
#     df = df.copy()
#     g  = df.groupby("case_id")

#     for col in core:
#         # Rolling windows: short (5), medium (15), long (30)
#         for w in [5, 15, 30]:
#             rm = g[col].transform(lambda x: x.rolling(w, min_periods=1).mean())
#             rs = g[col].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
#             df[f"{col}_rmean{w}"] = rm
#             df[f"{col}_rstd{w}"]  = rs
#             # Deviation from local mean in units of local std  (local z-score)
#             df[f"{col}_lz{w}"]    = (df[col] - rm) / (rs + 1e-6)

#         # Rate of change
#         df[f"{col}_d1"] = g[col].transform(lambda x: x.diff().fillna(0))
#         df[f"{col}_d2"] = g[col].transform(lambda x: x.diff().diff().fillna(0))

#         # Global z-score
#         mu, sd = df[col].mean(), df[col].std() + 1e-6
#         df[f"{col}_gz"] = (df[col] - mu) / sd   # keep sign — direction matters

#         # Exponential moving average deviation (faster reaction than rolling mean)
#         ema = g[col].transform(lambda x: x.ewm(span=10, adjust=False).mean())
#         df[f"{col}_ema_dev"] = df[col] - ema

#     # ── Clinical composite features ──────────────────────────────────────────
#     if "HR" in core and "MBP" in core:
#         # Shock index proxy: high HR + low MBP = critical
#         df["shock_proxy"] = df["HR"] / (df["MBP"] + 1e-6)

#     if "SpO2" in core and "HR" in core:
#         # O2 demand mismatch: low SpO2 + high HR = respiratory distress
#         df["o2_demand"]   = (100 - df["SpO2"]) * df["HR"] / 100

#     if "MBP" in core:
#         df["hypotension"]   = (df["MBP"] < 65).astype(float)   # clinical threshold
#         df["hypertension"]  = (df["MBP"] > 110).astype(float)

#     if "SpO2" in core:
#         df["hypoxia_sev"]  = np.where(df["SpO2"] < 85, 2.0,
#                              np.where(df["SpO2"] < 90, 1.0, 0.0))

#     if "HR" in core:
#         df["tachy"]  = (df["HR"] > 120).astype(float)
#         df["brady"]  = (df["HR"] < 50).astype(float)

#     # Count how many vitals are simultaneously abnormal (multi-organ signal)
#     flag_cols = [c for c in ["hypotension","hypoxia_sev","tachy","brady"] if c in df.columns]
#     if flag_cols:
#         df["multi_alarm"] = df[flag_cols].gt(0).sum(axis=1).astype(float)

#     return df

# print("\nEngineering features...")
# train = make_features(train, CORE)
# test  = make_features(test,  CORE)

# DROP     = ["case_id","time_sec"] + [c for c in ["label","anomaly","is_anomaly","target"] if c in train.columns]
# FEAT     = [c for c in train.columns if c not in DROP]
# print(f"Feature count: {len(FEAT)}")

# X_tr = train[FEAT].values.astype(np.float32)
# X_te = test[FEAT].values.astype(np.float32)

# scaler = RobustScaler()
# X_tr   = scaler.fit_transform(X_tr)
# X_te   = scaler.transform(X_te)

# # ── 6. ANOMALY MODELS (all O(n log n) or faster) ─────────────────────────────
# def norm01(arr):
#     lo, hi = arr.min(), arr.max()
#     return (arr - lo) / (hi - lo + 1e-9)

# tr_score = np.zeros(len(X_tr))
# te_score = np.zeros(len(X_te))

# # ── 6a. IsolationForest ───────────────────────────────────────────────────────
# print("[1/4] IsolationForest...")
# iso = IsolationForest(n_estimators=400, contamination=0.05,
#                       max_samples=min(50_000, len(X_tr)),
#                       random_state=42, n_jobs=-1)
# iso.fit(X_tr)
# # IMPORTANT: sklearn IForest decision_function → higher = more NORMAL → negate
# tr_score += 0.40 * norm01(-iso.decision_function(X_tr))
# te_score += 0.40 * norm01(-iso.decision_function(X_te))

# # ── 6b. HBOS ─────────────────────────────────────────────────────────────────
# print("[2/4] HBOS...")
# hbos = HBOS(n_bins=50, contamination=0.05)
# hbos.fit(X_tr)
# tr_score += 0.20 * norm01(hbos.decision_scores_)
# te_score += 0.20 * norm01(hbos.decision_function(X_te))

# # ── 6c. ECOD ─────────────────────────────────────────────────────────────────
# print("[3/4] ECOD...")
# ecod = ECOD(contamination=0.05, n_jobs=-1)
# ecod.fit(X_tr)
# tr_score += 0.25 * norm01(ecod.decision_scores_)
# te_score += 0.25 * norm01(ecod.decision_function(X_te))

# # ── 6d. MiniBatchKMeans — distance to nearest centroid ───────────────────────
# print("[4/4] MiniBatchKMeans (LOF substitute, O(n))...")
# km = MiniBatchKMeans(n_clusters=80, batch_size=10_000,
#                      random_state=42, n_init=5)
# km.fit(X_tr)

# def km_dist_chunked(X, km, chunk=50_000):
#     out = np.empty(len(X), dtype=np.float32)
#     for i in range(0, len(X), chunk):
#         Xc = X[i:i+chunk]
#         # shape (chunk, n_clusters) → take min distance
#         d  = np.linalg.norm(
#                  Xc[:, None, :] - km.cluster_centers_[None, :, :], axis=2
#              ).min(axis=1)
#         out[i:i+chunk] = d
#     return out

# tr_score += 0.15 * norm01(km_dist_chunked(X_tr, km))
# te_score += 0.15 * norm01(km_dist_chunked(X_te, km))

# # Final ensemble normalisation
# def final_norm(arr):
#     lo, hi = arr.min(), arr.max()
#     return (arr - lo) / (hi - lo + 1e-9)

# tr_score = final_norm(tr_score)
# te_score = final_norm(te_score)
# print("✅ Models done.")

# # ── 7. RULE-BASED HARD BOOST ─────────────────────────────────────────────────
# # Clinical alarm conditions always deserve high anomaly scores.
# # Directly improves Recall (15% of scoring).
# def rule_boost(df, base, core):
#     boost = np.zeros(len(df))
#     if "SpO2" in core:
#         boost += np.where(df["SpO2"] < 85, 0.50, 0)   # severe hypoxia
#         boost += np.where((df["SpO2"]>=85)&(df["SpO2"]<90), 0.25, 0)
#     if "MBP" in core:
#         boost += np.where(df["MBP"] < 50, 0.50, 0)    # severe hypotension
#         boost += np.where(df["MBP"] > 130, 0.30, 0)   # hypertensive crisis
#     if "HR" in core:
#         boost += np.where(df["HR"] > 150, 0.35, 0)
#         boost += np.where(df["HR"] < 40,  0.35, 0)
#     if "Temp" in core:
#         boost += np.where(df["Temp"] > 40.5, 0.30, 0)
#         boost += np.where(df["Temp"] < 35.0, 0.30, 0)
#     if "shock_proxy" in df.columns:
#         boost += np.where(df["shock_proxy"] > 2.0, 0.35, 0)
#     if "multi_alarm" in df.columns:
#         boost += np.where(df["multi_alarm"] >= 2, 0.40, 0)  # 2+ alarms simultaneously
#     return final_norm(np.clip(base + boost, 0, 1))

# print("Applying clinical rule boosts...")
# tr_score = rule_boost(train, tr_score, CORE)
# te_score = rule_boost(test,  te_score, CORE)

# # ── 8. QUICK SANITY CHECK ON TRAIN ───────────────────────────────────────────
# # We have no labels so we check score distribution — anomalies should be rare
# pct95 = np.percentile(tr_score, 95)
# pct99 = np.percentile(tr_score, 99)
# print(f"\nTrain score distribution:")
# print(f"  Mean : {tr_score.mean():.4f}")
# print(f"  Std  : {tr_score.std():.4f}")
# print(f"  P95  : {pct95:.4f}  ← ~5% flagged as anomaly")
# print(f"  P99  : {pct99:.4f}  ← ~1% flagged as anomaly")
# print(f"  Max  : {tr_score.max():.4f}")

# # ── 9. BUILD SUBMISSION ───────────────────────────────────────────────────────
# submission = test[["case_id","time_sec"]].copy()
# submission["anomaly_score"] = te_score

# assert len(submission) == len(test),               "❌ Row count mismatch"
# assert submission["anomaly_score"].between(0,1).all(), "❌ Scores out of [0,1]"
# assert not submission[["case_id","time_sec"]].duplicated().any(), "❌ Duplicate rows"

# print(f"\nSubmission: {submission.shape}")
# print(f"Score range: [{te_score.min():.4f}, {te_score.max():.4f}]")
# print(submission.head())

# submission.to_csv("yourname_submission.csv", index=False)
# print("\n✅ Saved: yourname_submission.csv")
# print("Columns:", submission.columns.tolist())

# # ── 10. ALSO SAVE AN INVERTED VERSION ─────────────────────────────────────────
# # If your leaderboard ROC-AUC is < 0.5, your scores are backwards.
# # In that case submit the _inverted version instead.
# inv = submission.copy()
# inv["anomaly_score"] = 1 - inv["anomaly_score"]
# inv.to_csv("yourname_submission_INVERTED.csv", index=False)
# print("✅ Also saved: yourname_submission_INVERTED.csv")
# print("   → If ROC-AUC < 0.5 on leaderboard, submit the INVERTED file!")

In [ ]:
# print(submission.columns.tolist())
# print(submission.head(3))
# print(submission['anomaly_score'].describe())


In [ ]:
# sub = pd.read_csv("/content/yourname_submission_INVERTED.csv")  # ← use your actual filename here
# test_df = pd.read_csv("test_vitals.csv")

# print("Test rows:      ", len(test_df))
# print("Submission rows:", len(sub))
# print("Missing rows:   ", len(test_df) - len(sub))

# test_keys = set(zip(test_df['case_id'], test_df['time_sec']))
# sub_keys  = set(zip(sub['case_id'],     sub['time_sec']))
# print("Rows in test but NOT in submission:", len(test_keys - sub_keys))
# print("Rows in submission but NOT in test:", len(sub_keys - test_keys))

# print("\nTest case_ids:      ", test_df['case_id'].nunique())
# print("Submission case_ids:", sub['case_id'].nunique())

In [ ]:
# # Check if case_id + time_sec ordering matches exactly
# test_df_sorted = test_df.sort_values(['case_id','time_sec']).reset_index(drop=True)
# sub_sorted     = sub.sort_values(['case_id','time_sec']).reset_index(drop=True)

# # Are the keys identical after sorting?
# keys_match = (test_df_sorted['case_id'] == sub_sorted['case_id']).all() and \
#              (test_df_sorted['time_sec'] == sub_sorted['time_sec']).all()
# print("Keys match after sort:", keys_match)

# # Check score distribution more carefully
# import numpy as np
# scores = sub['anomaly_score'].values
# print("\nScore distribution:")
# for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
#     print(f"  P{p:2d}: {np.percentile(scores, p):.4f}")

# print("\nHow many rows score > 0.5:", (scores > 0.5).sum())
# print("How many rows score > 0.9:", (scores > 0.9).sum())
# print("How many rows score = 1.0:", (scores == 1.0).sum())
# print("How many rows score = 0.0:", (scores == 0.0).sum())

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.preprocessing import RobustScaler
# from sklearn.ensemble import IsolationForest
# from pyod.models.hbos  import HBOS
# from pyod.models.ecod  import ECOD
# from sklearn.cluster import MiniBatchKMeans

# train = pd.read_csv("train_vitals.csv")
# test  = pd.read_csv("test_vitals.csv")

# # ── Rename columns ────────────────────────────────────────────────────────────
# rename = {}
# for col in train.columns:
#     cl = col.lower().strip()
#     if cl in ["hr","heart_rate"]:              rename[col]="HR"
#     elif cl in ["spo2","o2sat","oxygen"]:      rename[col]="SpO2"
#     elif cl in ["mbp","map","mean_bp"]:        rename[col]="MBP"
#     elif cl in ["temp","temperature"]:         rename[col]="Temp"
# train.rename(columns=rename, inplace=True)
# test.rename(columns=rename, inplace=True)

# CORE = ["HR","SpO2","MBP"]  # skip Temp — too sparse
# print("Vitals:", CORE)

# # ── Sort & clip ───────────────────────────────────────────────────────────────
# for df in [train, test]:
#     df.sort_values(["case_id","time_sec"], inplace=True)
#     df.reset_index(drop=True, inplace=True)

# BOUNDS = {"HR":(20,300), "SpO2":(50,100), "MBP":(20,200)}
# for df in [train, test]:
#     for col,(lo,hi) in BOUNDS.items():
#         if col in df.columns:
#             df[col] = df[col].clip(lo,hi)

# # ── Impute ────────────────────────────────────────────────────────────────────
# for df in [train, test]:
#     df[CORE] = df.groupby("case_id")[CORE].transform(lambda g: g.ffill().bfill())
#     df[CORE] = df[CORE].fillna(df[CORE].median())

# # ── Features ──────────────────────────────────────────────────────────────────
# def make_features(df, core):
#     df = df.copy()
#     g  = df.groupby("case_id")
#     for col in core:
#         for w in [5, 15, 30]:
#             rm = g[col].transform(lambda x: x.rolling(w, min_periods=1).mean())
#             rs = g[col].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
#             df[f"{col}_lz{w}"] = (df[col] - rm) / (rs + 1e-6)  # local z-score only
#         df[f"{col}_d1"]  = g[col].transform(lambda x: x.diff().fillna(0))
#         df[f"{col}_d2"]  = g[col].transform(lambda x: x.diff().diff().fillna(0))
#         mu,sd = df[col].mean(), df[col].std()+1e-6
#         df[f"{col}_gz"]  = (df[col] - mu) / sd
#         ema = g[col].transform(lambda x: x.ewm(span=10,adjust=False).mean())
#         df[f"{col}_ema_dev"] = df[col] - ema
#     if "HR" in core and "MBP" in core:
#         df["shock_proxy"] = df["HR"] / (df["MBP"] + 1e-6)
#     if "SpO2" in core and "HR" in core:
#         df["o2_demand"] = (100 - df["SpO2"]) * df["HR"] / 100
#     return df

# print("Engineering features...")
# train = make_features(train, CORE)
# test  = make_features(test,  CORE)

# DROP  = ["case_id","time_sec", "Temp"] # Explicitly drop 'Temp' as it's not imputed
# FEAT  = [c for c in train.columns if c not in DROP]
# print(f"Features: {len(FEAT)}")

# X_tr = train[FEAT].values.astype(np.float32)
# X_te = test[FEAT].values.astype(np.float32)

# scaler = RobustScaler()
# X_tr   = scaler.fit_transform(X_tr)
# X_te   = scaler.transform(X_te)

# # ── Models ────────────────────────────────────────────────────────────────────
# def norm01(a): return (a-a.min())/(a.max()-a.min()+1e-9)

# tr_s = np.zeros(len(X_tr))
# te_s = np.zeros(len(X_te))

# print("[1/4] IsolationForest...")
# iso = IsolationForest(n_estimators=400, contamination=0.05,
#                       max_samples=min(50_000,len(X_tr)),
#                       random_state=42, n_jobs=-1)
# iso.fit(X_tr)
# tr_s += 0.40 * norm01(-iso.decision_function(X_tr))
# te_s += 0.40 * norm01(-iso.decision_function(X_te))

# print("[2/4] HBOS...")
# hbos = HBOS(n_bins=50, contamination=0.05)
# hbos.fit(X_tr)
# tr_s += 0.25 * norm01(hbos.decision_scores_)
# te_s += 0.25 * norm01(hbos.decision_function(X_te))

# print("[3/4] ECOD...")
# ecod = ECOD(contamination=0.05, n_jobs=-1)
# ecod.fit(X_tr)
# tr_s += 0.25 * norm01(ecod.decision_scores_)
# te_s += 0.25 * norm01(ecod.decision_function(X_te))

# print("[4/4] MiniBatchKMeans...")
# km = MiniBatchKMeans(n_clusters=80, batch_size=10_000, random_state=42, n_init=5)
# km.fit(X_tr)

# def km_dist(X, km, chunk=50_000):
#     out = np.empty(len(X), dtype=np.float32)
#     for i in range(0, len(X), chunk):
#         Xc = X[i:i+chunk]
#         out[i:i+chunk] = np.linalg.norm(
#             Xc[:,None,:] - km.cluster_centers_[None,:,:], axis=2).min(axis=1)
#     return out

# tr_s += 0.10 * norm01(km_dist(X_tr, km))
# te_s += 0.10 * norm01(km_dist(X_te, km))

# # ── Final normalise — NO rule boost (was causing inflation) ───────────────────
# te_s = norm01(te_s)
# tr_s = norm01(tr_s)

# # ── Verify distribution before submitting ─────────────────────────────────────
# print("\n=== SCORE DISTRIBUTION (should be right-skewed, most rows LOW) ===")
# for p in [25,50,75,90,95,99]:
#     print(f"  P{p}: {np.percentile(te_s,p):.4f}")
# print(f"  Rows > 0.5: {(te_s>0.5).sum()} / {len(te_s)} = {(te_s>0.5).mean()*100:.1f}%")
# print(f"  Rows > 0.9: {(te_s>0.9).sum()} / {len(te_s)} = {(te_s>0.9).mean()*100:.1f}%")

# # ── Save ──────────────────────────────────────────────────────────────────────
# submission = test[["case_id","time_sec"]].copy()
# submission["anomaly_score"] = te_s
# submission.to_csv("abduttaiyeb_submission.csv", index=False)
# print("\n✅ Saved! Rows:", len(submission))

In [ ]:
# print("Train score dist:")
# for p in [25,50,75,90,95,99]:
#     print(f"  P{p}: {np.percentile(tr_s,p):.4f}")
# print(f"  Rows > 0.5: {(tr_s>0.5).mean()*100:.1f}%")

In [ ]:
# # Invert and save as separate file
# inv = submission.copy()
# inv["anomaly_score"] = 1 - inv["anomaly_score"]
# inv.to_csv("abduttaiyeb_submission_inv.csv", index=False)

# print("Inverted distribution:")
# for p in [25,50,75,90,95,99]:
#     print(f"  P{p}: {np.percentile(inv['anomaly_score'],p):.4f}")

In [ ]:
# # Merge scores back with original vitals to see what's being flagged
# check = test[["case_id","time_sec","HR","SpO2","MBP"]].copy()
# check["anomaly_score"] = te_s

# print("=== TOP 20 highest scored rows ===")
# print(check.nlargest(20, "anomaly_score")[["case_id","time_sec","HR","SpO2","MBP","anomaly_score"]].to_string())

# print("\n=== TOP 20 lowest scored rows ===")
# print(check.nsmallest(20, "anomaly_score")[["case_id","time_sec","HR","SpO2","MBP","anomaly_score"]].to_string())

In [ ]:
# # What does the raw data actually look like?
# print("=== NULL % per column ===")
# print(train.isnull().mean() * 100)

# print("\n=== Value counts sample — how often are readings exactly repeated? ===")
# for col in ["HR", "SpO2", "MBP"]:
#     print(f"\n{col} — most common values:")
#     print(train[col].value_counts().head(10))

# print("\n=== Are there sudden zero readings? ===")
# for col in ["HR", "SpO2", "MBP"]:
#     print(f"{col} == 0: {(train[col]==0).sum()}")

# print("\n=== Consecutive duplicate rows per patient? ===")
# for col in ["HR", "SpO2", "MBP"]:
#     dupes = train.groupby("case_id")[col].apply(
#         lambda x: (x == x.shift()).sum()
#     ).sum()
#     print(f"{col} consecutive duplicates: {dupes}")

In [ ]:
# # ── 0. INSTALLS ───────────────────────────────────────────────────────────────
# import subprocess, sys
# def pip(pkg): subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
# pip("pyod")

# import numpy as np
# import pandas as pd

# train = pd.read_csv("train_vitals.csv")
# test  = pd.read_csv("test_vitals.csv")

# # Rename
# rename = {}
# for col in train.columns:
#     cl = col.lower().strip()
#     if cl == "hr":             rename[col] = "HR"
#     elif cl == "spo2":         rename[col] = "SpO2"
#     elif cl in ["mbp","map"]:  rename[col] = "MBP"
#     elif cl == "temp":         rename[col] = "Temp"
# train.rename(columns=rename, inplace=True)
# test.rename(columns=rename, inplace=True)

# for df in [train, test]:
#     df.sort_values(["case_id","time_sec"], inplace=True)
#     df.reset_index(drop=True, inplace=True)

# CORE = ["HR", "SpO2", "MBP"]

# def make_artifact_features(df, core):
#     df = df.copy()
#     g  = df.groupby("case_id")

#     for col in core:
#         # ── 1. FLAT-LINE LENGTH ───────────────────────────────────────────────
#         # How many consecutive identical readings has this sensor been stuck on?
#         def flatline_len(x):
#             x = x.reset_index(drop=True)
#             out = np.ones(len(x))
#             count = 1
#             for i in range(1, len(x)):
#                 if x[i] == x[i-1]:
#                     count += 1
#                 else:
#                     count = 1
#                 out[i] = count
#             return pd.Series(out, dtype=np.float32)

#         df[f"{col}_flat"] = g[col].transform(flatline_len)

#         # ── 2. SENSOR FLOOR/CEILING HIT ──────────────────────────────────────
#         # MBP=20 is the clip floor = sensor dropout signal
#         # SpO2=100 for long stretches = likely saturated/artifact
#         col_min = df[col].min()
#         col_max = df[col].max()
#         df[f"{col}_at_floor"]   = (df[col] == col_min).astype(float)
#         df[f"{col}_at_ceiling"] = (df[col] == col_max).astype(float)

#         # ── 3. SUDDEN JUMP (spike artifact) ──────────────────────────────────
#         d1 = g[col].transform(lambda x: x.diff().abs().fillna(0))
#         d2 = g[col].transform(lambda x: x.diff().diff().abs().fillna(0))
#         df[f"{col}_jump1"] = d1
#         df[f"{col}_jump2"] = d2

#         # ── 4. LOCAL Z-SCORE (still useful for spikes) ────────────────────────
#         for w in [5, 20]:
#             rm = g[col].transform(lambda x: x.rolling(w, min_periods=1).mean())
#             rs = g[col].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
#             df[f"{col}_lz{w}"] = (df[col] - rm) / (rs + 1e-6)

#         # ── 5. FLATLINE + EXTREME VALUE COMBO ────────────────────────────────
#         # Sensor stuck at floor = strongest artifact signal
#         df[f"{col}_flat_floor"] = df[f"{col}_flat"] * df[f"{col}_at_floor"]
#         df[f"{col}_flat_ceil"]  = df[f"{col}_flat"] * df[f"{col}_at_ceiling"]

#     # ── 6. CROSS-CHANNEL ARTIFACT ─────────────────────────────────────────────
#     # All channels flat simultaneously = monitor disconnect
#     flat_cols = [f"{c}_flat" for c in core]
#     df["all_flat"]   = df[flat_cols].min(axis=1)   # min = all must be flat
#     df["total_flat"] = df[flat_cols].sum(axis=1)   # sum = how many are flat

#     return df

# print("Building artifact features...")
# train = make_artifact_features(train, CORE)
# test  = make_artifact_features(test,  CORE)

# DROP  = ["case_id","time_sec","Temp"]
# FEAT  = [c for c in train.columns if c not in DROP and c not in CORE]
# print(f"Features: {len(FEAT)}")

# X_tr = train[FEAT].values.astype(np.float32)
# X_te = test[FEAT].values.astype(np.float32)

# from sklearn.preprocessing import RobustScaler
# scaler = RobustScaler()
# X_tr   = scaler.fit_transform(X_tr)
# X_te   = scaler.transform(X_te)

# # Fill any remaining NaNs after scaling
# X_tr = np.nan_to_num(X_tr, nan=0.0)
# X_te = np.nan_to_num(X_te, nan=0.0)

# from sklearn.ensemble import IsolationForest
# from pyod.models.hbos import HBOS
# from pyod.models.ecod import ECOD

# def norm01(a): return (a - a.min()) / (a.max() - a.min() + 1e-9)

# tr_s = np.zeros(len(X_tr))
# te_s = np.zeros(len(X_te))

# print("[1/3] IsolationForest...")
# iso = IsolationForest(n_estimators=400, contamination=0.05,
#                       max_samples=min(50_000, len(X_tr)),
#                       random_state=42, n_jobs=-1)
# iso.fit(X_tr)
# tr_s += 0.40 * norm01(-iso.decision_function(X_tr))
# te_s += 0.40 * norm01(-iso.decision_function(X_te))

# print("[2/3] HBOS...")
# hbos = HBOS(n_bins=50, contamination=0.05)
# hbos.fit(X_tr)
# tr_s += 0.30 * norm01(hbos.decision_scores_)
# te_s += 0.30 * norm01(hbos.decision_function(X_te))

# print("[3/3] ECOD...")
# ecod = ECOD(contamination=0.05, n_jobs=-1)
# ecod.fit(X_tr)
# tr_s += 0.30 * norm01(ecod.decision_scores_)
# te_s += 0.30 * norm01(ecod.decision_function(X_te))

# te_s = norm01(te_s)
# tr_s = norm01(tr_s)

# # ── RULE SCORES (artifact-specific, not clinical) ─────────────────────────────
# rule = np.zeros(len(test))

# # MBP=20 flatline = almost certain sensor dropout
# rule += np.where(test["MBP"] == 20, 0.6, 0)

# # Long flatlines on any channel
# for col in CORE:
#     flat = test[f"{col}_flat"].values
#     rule += np.where(flat > 10, 0.3, 0)
#     rule += np.where(flat > 30, 0.3, 0)   # extra boost for very long flatlines

# # All channels flat simultaneously
# rule += np.where(test["all_flat"].values > 5, 0.4, 0)

# # Blend: 50% model + 50% artifact rules
# te_s = norm01(0.5 * te_s + 0.5 * norm01(rule))

# # ── Verify ────────────────────────────────────────────────────────────────────
# print("\n=== SCORE DISTRIBUTION ===")
# for p in [25, 50, 75, 90, 95, 99]:
#     print(f"  P{p}: {np.percentile(te_s, p):.4f}")
# print(f"  Rows > 0.5: {(te_s>0.5).sum()} ({(te_s>0.5).mean()*100:.1f}%)")

# # ── What are we flagging now? ─────────────────────────────────────────────────
# check = test[["case_id","time_sec","HR","SpO2","MBP"]].copy()
# check["flat_HR"]   = test["HR_flat"].values
# check["flat_SpO2"] = test["SpO2_flat"].values
# check["flat_MBP"]  = test["MBP_flat"].values
# check["score"]     = te_s

# print("\n=== TOP 15 ANOMALIES ===")
# print(check.nlargest(15,"score")[
#     ["case_id","time_sec","HR","SpO2","MBP","flat_HR","flat_SpO2","flat_MBP","score"]
# ].to_string())

# # ── Save ──────────────────────────────────────────────────────────────────────
# submission = test[["case_id","time_sec"]].copy()
# submission["anomaly_score"] = te_s
# submission.to_csv("abduttaiyeb_submission.csv", index=False)
# print(f"\n✅ Saved! Rows: {len(submission)}")

In [ ]:
# print("=== SCORE DISTRIBUTION ===")
# for p in [25, 50, 75, 90, 95, 99]:
#     print(f"  P{p}: {np.percentile(te_s, p):.4f}")
# print(f"  Rows > 0.5: {(te_s>0.5).sum()} ({(te_s>0.5).mean()*100:.1f}%)")

# print("\n=== TOP 15 ANOMALIES ===")
# print(check.nlargest(15,"score")[
#     ["case_id","time_sec","HR","SpO2","MBP","flat_HR","flat_SpO2","flat_MBP","score"]
# ].to_string())

# print("\n=== MBP=20 rows count ===")
# print((test["MBP"] == 20).sum())

# print("\n=== Longest flatlines ===")
# for col in ["HR","SpO2","MBP"]:
#     print(f"  {col} max flat run: {test[f'{col}_flat'].max()}")

In [ ]:
# # Check raw data before clipping
# print("MBP negative values:", (test["MBP"] < 0).sum())
# print("MBP < 20 values:", (test["MBP"] < 20).sum())
# print("SpO2 > 100:", (test["SpO2"] > 100).sum())
# print("\nMBP min/max:", test["MBP"].min(), test["MBP"].max())
# print("HR min/max:", test["HR"].min(), test["HR"].max())
# print("SpO2 min/max:", test["SpO2"].min(), test["SpO2"].max())

# # Also check train raw
# train_raw = pd.read_csv("train_vitals.csv")
# print("\n=== TRAIN RAW ===")
# print("MBP < 0:", (train_raw["mbp"] < 0).sum())
# print("MBP < 20:", (train_raw["mbp"] < 20).sum())
# print("SpO2 max flat in train:", )

In [ ]:
# train_raw = pd.read_csv("train_vitals.csv")
# test_raw  = pd.read_csv("test_vitals.csv")

# # Check exact column names first
# print(train_raw.columns.tolist())
# print(test_raw.columns.tolist())

In [ ]:
# # Run this and paste output
# print("=== TRAIN vs TEST anomaly rate check ===")
# print("Train rows:", len(train_raw))
# print("Test rows:", len(test_raw))

# # How many rows have out-of-range values in test?
# for col, (lo,hi) in [("HR",(20,300)),("SpO2",(50,100)),("MBP",(20,200))]:
#     oor = ((test_raw[col] < lo) | (test_raw[col] > hi)).sum()
#     print(f"  {col} out-of-range: {oor} ({oor/len(test_raw)*100:.2f}%)")

# # Flatline distribution
# print("\nSpO2=100 consecutive run lengths (test):")
# from collections import Counter
# runs = []
# for cid, grp in test_raw.groupby("case_id"):
#     s = grp["SpO2"].values
#     c = 1
#     for i in range(1, len(s)):
#         if s[i] == s[i-1]:
#             c += 1
#         else:
#             runs.append(c)
#             c = 1
#     runs.append(c)

# runs = np.array(runs)
# print(f"  Total runs: {len(runs)}")
# print(f"  Runs > 10:  {(runs>10).sum()}")
# print(f"  Runs > 50:  {(runs>50).sum()}")
# print(f"  Runs > 200: {(runs>200).sum()}")
# print(f"  Max run:    {runs.max()}")

In [ ]:
# import numpy as np
# import pandas as pd

# train_raw = pd.read_csv("train_vitals.csv")
# test_raw  = pd.read_csv("test_vitals.csv")

# for df in [train_raw, test_raw]:
#     df.sort_values(["case_id","time_sec"], inplace=True)
#     df.reset_index(drop=True, inplace=True)

# def score_it(df):
#     score = np.zeros(len(df))

#     # MBP out of range — strongest signal (5.98% of rows)
#     mbp = df["MBP"].values
#     score += np.where(mbp < 20,  1.0, 0)   # below floor
#     score += np.where(mbp > 200, 1.0, 0)   # above ceiling
#     # Graduated severity
#     score += np.where(mbp < 10,  1.0, 0)   # extra boost for extreme
#     score += np.where(mbp < 0,   1.0, 0)   # negative = definite artifact

#     # HR out of range — secondary signal (0.20% of rows)
#     hr = df["HR"].values
#     score += np.where(hr < 20,  0.8, 0)
#     score += np.where(hr > 300, 0.8, 0)

#     # MBP sudden huge jump (spike artifact)
#     g = df.groupby("case_id")
#     mbp_jump = g["MBP"].transform(lambda x: x.diff().abs().fillna(0)).values
#     hr_jump  = g["HR"].transform(lambda x: x.diff().abs().fillna(0)).values

#     score += np.where(mbp_jump > 50, 0.5, 0)
#     score += np.where(mbp_jump > 100, 0.5, 0)
#     score += np.where(hr_jump  > 50, 0.3, 0)

#     # MBP flatline at floor (sensor dropout)
#     def flat_len(x):
#         x = x.reset_index(drop=True)
#         out = np.ones(len(x))
#         c = 1
#         for i in range(1, len(x)):
#             c = c+1 if x[i]==x[i-1] else 1
#             out[i] = c
#         return pd.Series(out, dtype=np.float32)

#     mbp_flat = g["MBP"].transform(flat_len).values
#     hr_flat  = g["HR"].transform(flat_len).values

#     # Only flag MBP flatlines if MBP is also at an extreme value
#     score += np.where((mbp_flat > 10) & (mbp < 20), 0.5, 0)
#     score += np.where((hr_flat  > 10) & (hr  < 20), 0.4, 0)

#     # Normalize to [0,1]
#     score = (score - score.min()) / (score.max() - score.min() + 1e-9)
#     return score

# print("Scoring...")
# test_score = score_it(test_raw)

# print("\n=== DISTRIBUTION ===")
# for p in [25,50,75,90,95,99]:
#     print(f"  P{p}: {np.percentile(test_score,p):.4f}")
# print(f"  Rows > 0.5: {(test_score>0.5).sum()} ({(test_score>0.5).mean()*100:.2f}%)")
# print(f"  Rows = 0.0: {(test_score==0.0).sum()} ({(test_score==0.0).mean()*100:.2f}%)")

# check = test_raw[["case_id","time_sec","HR","SpO2","MBP"]].copy()
# check["score"] = test_score
# print("\n=== TOP 20 ===")
# print(check.nlargest(20,"score")[["case_id","time_sec","HR","SpO2","MBP","score"]].to_string())
# print("\n=== BOTTOM 10 (should be perfectly normal) ===")
# print(check.nsmallest(10,"score")[["case_id","time_sec","HR","SpO2","MBP","score"]].to_string())

# sub = test_raw[["case_id","time_sec"]].copy()
# sub["anomaly_score"] = test_score
# sub.to_csv("abduttaiyeb_submission.csv", index=False)
# print(f"\n✅ Saved! Rows: {len(sub)}")

In [ ]:
# # How many unique MBP negative values exist?
# print("MBP negative breakdown:")
# print(test_raw[test_raw["MBP"] < 0]["MBP"].value_counts().sort_index())

# print("\nMBP < 20 breakdown:")
# print(test_raw[test_raw["MBP"] < 20]["MBP"].value_counts().sort_index())

# print("\nTotal flagged rows by case_id:")
# flagged = test_raw[test_score > 0.5]
# print(flagged["case_id"].value_counts())

In [ ]:
# import numpy as np
# import pandas as pd

# train_raw = pd.read_csv("train_vitals.csv")
# test_raw  = pd.read_csv("test_vitals.csv")

# for df in [train_raw, test_raw]:
#     df.sort_values(["case_id","time_sec"], inplace=True)
#     df.reset_index(drop=True, inplace=True)

# # Check if train has a label column we missed
# print("TRAIN COLUMNS:", train_raw.columns.tolist())
# print("Any column with <10 unique values (possible label):")
# for col in train_raw.columns:
#     u = train_raw[col].nunique()
#     if u < 10:
#         print(f"  {col}: {u} unique values → {train_raw[col].value_counts().to_dict()}")

# # Check if there's a separate label file
# import os
# print("\nAll files in current directory:")
# for f in os.listdir("."):
#     print(f"  {f}")

In [ ]:
import numpy as np
import pandas as pd

df_train = pd.read_csv("train_vitals.csv")
df_test  = pd.read_csv("test_vitals.csv")

for df in [df_train, df_test]:
    df.sort_values(["case_id","time_sec"], inplace=True)
    df.reset_index(drop=True, inplace=True)

THRESHOLDS = {
    'HR':   {'low': 40,   'high': 150},
    'MBP':  {'low': 50,   'high': 120},
    'SpO2': {'low': 90,   'high': 100},
    'Temp': {'low': 35.0, 'high': 39.5},
}

FEATURES = ['HR', 'MBP', 'SpO2', 'Temp']

# Exactly what starter does — treat <=0 as NaN
for df in [df_train, df_test]:
    df[FEATURES] = df[FEATURES].where(df[FEATURES] > 0)
    df[FEATURES] = df.groupby('case_id')[FEATURES].transform(
        lambda x: x.fillna(x.median())
    )
    df[FEATURES] = df[FEATURES].fillna(df[FEATURES].median())

# ── Threshold breach score ────────────────────────────────────────────────────
def threshold_score(df):
    score = np.zeros(len(df))
    for col, t in THRESHOLDS.items():
        if col not in df.columns:
            continue
        v = df[col].values
        # Below low threshold
        below = np.maximum(0, t['low']  - v) / t['low']
        # Above high threshold
        above = np.maximum(0, v - t['high']) / t['high']
        score += below + above
    return score

# ── Also use IForest exactly like starter but fixed (negate scores) ───────────
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

scaler  = StandardScaler()
X_train = scaler.fit_transform(df_train[FEATURES])
X_test  = scaler.transform(df_test[FEATURES])

print("Fitting IsolationForest...")
model = IsolationForest(n_estimators=300, contamination=0.1, random_state=42, n_jobs=-1)
model.fit(X_train)

raw   = model.decision_function(X_test)
# Starter's exact formula — higher = more anomalous
iforest_score = 1 - (raw - raw.min()) / (raw.max() - raw.min())

# Threshold breach score
thr_score = threshold_score(df_test)
thr_norm  = (thr_score - thr_score.min()) / (thr_score.max() - thr_score.min() + 1e-9)

# Blend — threshold score is more precise, iforest catches edge cases
final = 0.6 * thr_norm + 0.4 * iforest_score
final = (final - final.min()) / (final.max() - final.min() + 1e-9)

print("\n=== DISTRIBUTION ===")
for p in [25,50,75,90,95,99]:
    print(f"  P{p}: {np.percentile(final,p):.4f}")
print(f"  Rows > 0.5: {(final>0.5).sum()} ({(final>0.5).mean()*100:.1f}%)")

check = df_test[["case_id","time_sec","HR","MBP","SpO2","Temp"]].copy()
check["score"] = final
print("\n=== TOP 15 ===")
print(check.nlargest(15,"score")[["case_id","time_sec","HR","MBP","SpO2","Temp","score"]].to_string())

sub = df_test[["case_id","time_sec"]].copy()
sub["anomaly_score"] = final
sub.to_csv("abduttaiyeb_submission.csv", index=False)
print(f"\n✅ Saved! Rows: {len(sub)}")

Fitting IsolationForest...

=== DISTRIBUTION ===
  P25: 0.0179
  P50: 0.0412
  P75: 0.0794
  P90: 0.1610
  P95: 0.2363
  P99: 0.6057
  Rows > 0.5: 2922 (2.4%)

=== TOP 15 ===
       case_id  time_sec     HR    MBP   SpO2       Temp     score
87457      156      1660   55.0  331.0   99.0  35.900002  1.000000
87458      156      1662   55.0  331.0   99.0  35.900002  1.000000
87459      156      1664   55.0  331.0   99.0  35.900002  1.000000
87460      156      1666   55.0  331.0   99.0  35.900002  1.000000
87463      156      1672   54.0  330.0   99.0  35.900002  0.999141
87464      156      1674   54.0  330.0   99.0  35.900002  0.999141
87462      156      1670   54.0  328.0   99.0  35.900002  0.992724
87461      156      1668   55.0  328.0   99.0  35.900002  0.990374
87465      156      1676   54.0  321.0   99.0  35.900002  0.970264
87466      156      1678   54.0  321.0   99.0  35.900002  0.970264
15841      137      5740  306.0  109.0  100.0  20.400000  0.968129
87455      156      1

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score

def evaluate_submission(submission_csv, vitals_csv, threshold_percentile=90):
    sub = pd.read_csv(submission_csv)
    df  = pd.read_csv(vitals_csv)

    # merge on case_id + time_sec
    merged = df.merge(sub, on=["case_id","time_sec"])

    # pseudo labels from threshold breaches
    THRESHOLDS = {
        'HR':   {'low': 40,   'high': 150},
        'MBP':  {'low': 50,   'high': 120},
        'SpO2': {'low': 90,   'high': 100},
        'Temp': {'low': 35.0, 'high': 39.5},
    }
    y_true = np.zeros(len(merged))
    for col, t in THRESHOLDS.items():
        if col not in merged.columns: continue
        y_true += (merged[col] < t['low']).astype(int)
        y_true += (merged[col] > t['high']).astype(int)
    y_true = (y_true > 0).astype(int)

    y_score = merged["anomaly_score"].values
    y_score = (y_score - y_score.min()) / (y_score.max() - y_score.min() + 1e-9)
    y_pred  = (y_score >= np.percentile(y_score, threshold_percentile)).astype(int)

    roc = roc_auc_score(y_true, y_score)
    pr  = average_precision_score(y_true, y_score)
    f1  = f1_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)

    print(f"Pseudo-anomaly rate : {y_true.mean()*100:.2f}%")
    print(f"ROC-AUC : {roc:.4f}")
    print(f"PR-AUC  : {pr:.4f}")
    print(f"F1      : {f1:.4f}")
    print(f"Recall  : {rec:.4f}")
    print(f"─────────────────────")
    print(f"FINAL   : {0.40*roc + 0.30*pr + 0.15*f1 + 0.15*rec:.4f}")

# Usage — just pass your file names
evaluate_submission("abduttaiyeb_submission (1).csv", "test_vitals.csv")

Pseudo-anomaly rate : 12.27%
ROC-AUC : 0.7565
PR-AUC  : 0.5727
F1      : 0.2186
Recall  : 1.0000
─────────────────────
FINAL   : 0.6572


In [ ]:
# Look at one full case in detail
case_id = df_train["case_id"].iloc[0]
case = df_train[df_train["case_id"] == case_id].copy()

print(f"Case {case_id}: {len(case)} rows")
print(f"Time range: {case['time_sec'].min()} to {case['time_sec'].max()} seconds")
print(f"= {case['time_sec'].max()/3600:.1f} hours of surgery")
print()

# Check time gaps — missing time intervals = anomaly windows?
case["time_gap"] = case["time_sec"].diff()
print("Time gaps between readings:")
print(case["time_gap"].value_counts().head(10))
print(f"\nMax time gap: {case['time_gap'].max()} seconds")
print(f"Normal gap should be: 2 seconds (VitalDB records every 2 sec)")

# Large gaps = missing data = anomaly?
big_gaps = case[case["time_gap"] > 10]
print(f"\nRows with gap > 10 sec: {len(big_gaps)}")
print(big_gaps[["case_id","time_sec","time_gap","HR","MBP","SpO2"]].head(10))

Case 1: 5440 rows
Time range: 1 to 10910 seconds
= 3.0 hours of surgery

Time gaps between readings:
time_gap
2.0     5419
3.0        8
1.0        7
4.0        3
14.0       2
Name: count, dtype: int64

Max time gap: 14.0 seconds
Normal gap should be: 2 seconds (VitalDB records every 2 sec)

Rows with gap > 10 sec: 2
      case_id  time_sec  time_gap    HR   MBP   SpO2
15          1        43      14.0  78.0  -8.0   96.0
1150        1      2332      14.0  76.0 -70.0  100.0


In [ ]:
# Check this pattern across entire train
print("=== TIME GAP ANALYSIS ACROSS ALL CASES ===")
df_train_sorted = df_train.sort_values(["case_id","time_sec"]).reset_index(drop=True)
df_train_sorted["time_gap"] = df_train_sorted.groupby("case_id")["time_sec"].diff()

print("Time gap distribution:")
print(df_train_sorted["time_gap"].value_counts().head(15))

print(f"\nRows with gap > 10 sec: {(df_train_sorted['time_gap'] > 10).sum()}")
print(f"Rows with gap > 6 sec:  {(df_train_sorted['time_gap'] > 6).sum()}")

# What are MBP values around these gaps?
big_gap_rows = df_train_sorted[df_train_sorted["time_gap"] > 6]
print(f"\nMBP < 0 in big gap rows: {(big_gap_rows['MBP'] < 0).sum()}")
print(f"MBP < 0 in all rows:     {(df_train_sorted['MBP'] < 0).sum()}")

print("\nSample big gap rows:")
print(big_gap_rows[["case_id","time_sec","time_gap","HR","MBP","SpO2","Temp"]].head(20).to_string())

# Also check rows BEFORE the gap
print("\n=== MBP < 0 always coincides with big gaps? ===")
neg_mbp = df_train_sorted[df_train_sorted["MBP"] < 0]
print(f"Total negative MBP rows: {len(neg_mbp)}")
print(f"Their time gaps:")
print(neg_mbp["time_gap"].value_counts().head(10))

=== TIME GAP ANALYSIS ACROSS ALL CASES ===
Time gap distribution:
time_gap
2.0     264245
3.0        335
1.0        145
4.0         82
6.0         26
18.0        13
14.0        11
5.0         11
12.0        10
10.0         9
28.0         8
8.0          8
22.0         7
34.0         7
16.0         7
Name: count, dtype: int64

Rows with gap > 10 sec: 162
Rows with gap > 6 sec:  180

MBP < 0 in big gap rows: 16
MBP < 0 in all rows:     6735

Sample big gap rows:
       case_id  time_sec  time_gap    HR    MBP   SpO2       Temp
15           1        43      14.0  78.0   -8.0   96.0        NaN
1150         1      2332      14.0  76.0  -70.0  100.0  35.400002
14067        4     17276      21.0  86.0   70.0  100.0  35.400002
15626        7        21      18.0  67.0    2.0   65.0        NaN
15633        7        85      52.0  69.0   -2.0   62.0        NaN
15650        7       133      16.0  69.0    1.0   53.0        NaN
17275        7      3497     102.0  59.0   97.0  100.0  36.400002
17280   

In [ ]:
# Negative MBP = anomaly hypothesis
print("=== NEGATIVE MBP ANALYSIS ===")
print(f"Total train rows:      {len(df_train_sorted)}")
print(f"Negative MBP rows:     {(df_train_sorted['MBP'] < 0).sum()}")
print(f"Anomaly rate:          {(df_train_sorted['MBP'] < 0).mean()*100:.2f}%")

print("\nNegative MBP value distribution:")
print(df_train_sorted[df_train_sorted['MBP'] < 0]['MBP'].describe())

print("\nWhen MBP is negative, what are HR and SpO2?")
neg = df_train_sorted[df_train_sorted['MBP'] < 0]
print(f"HR mean:   {neg['HR'].mean():.1f}")
print(f"SpO2 mean: {neg['SpO2'].mean():.1f}")
print(f"MBP mean:  {neg['MBP'].mean():.1f}")

print("\nSample negative MBP rows:")
print(neg[["case_id","time_sec","HR","MBP","SpO2","Temp"]].head(20).to_string())

# Check test too
print(f"\n=== TEST negative MBP ===")
print(f"Negative MBP in test: {(df_test['MBP'] < 0).sum()}")
print(f"Rate: {(df_test['MBP'] < 0).mean()*100:.2f}%")


=== NEGATIVE MBP ANALYSIS ===
Total train rows:      265058
Negative MBP rows:     6735
Anomaly rate:          2.54%

Negative MBP value distribution:
count    6735.000000
mean      -10.454937
std        10.252341
min       -70.000000
25%       -12.000000
50%        -7.000000
75%        -4.000000
max        -1.000000
Name: MBP, dtype: float64

When MBP is negative, what are HR and SpO2?
HR mean:   76.9
SpO2 mean: 98.4
MBP mean:  -10.5

Sample negative MBP rows:
    case_id  time_sec    HR  MBP  SpO2  Temp
0         1         1  88.0 -8.0  96.0   NaN
1         1         3  87.0 -8.0  96.0   NaN
2         1         5  88.0 -8.0  97.0   NaN
3         1         7  88.0 -9.0  96.0   NaN
4         1         9  88.0 -9.0  96.0   NaN
5         1        11  88.0 -8.0  96.0   NaN
6         1        13  89.0 -8.0  97.0   NaN
7         1        15  90.0 -9.0  97.0   NaN
8         1        17  90.0 -9.0  97.0   NaN
9         1        19  79.0 -9.0  97.0   NaN
10        1        21  79.0 -9.0  97.0 

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score

def evaluate(vitals_csv, submission_csv):
    df  = pd.read_csv(vitals_csv)
    sub = pd.read_csv(submission_csv)
    merged = df.merge(sub, on=["case_id","time_sec"])

    y_true  = (merged["MBP"] < 0).astype(int).values
    y_score = merged["anomaly_score"].values
    y_pred  = (y_score >= 0.5).astype(int)

    roc = roc_auc_score(y_true, y_score)
    pr  = average_precision_score(y_true, y_score)
    f1  = f1_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)

    print(f"ROC-AUC : {roc:.4f}")
    print(f"PR-AUC  : {pr:.4f}")
    print(f"F1      : {f1:.4f}")
    print(f"Recall  : {rec:.4f}")
    print(f"─────────────────")
    print(f"FINAL   : {0.40*roc + 0.30*pr + 0.15*f1 + 0.15*rec:.4f}")

evaluate("test_vitals.csv", "abduttaiyeb_submission (1).csv")

ROC-AUC : 1.0000
PR-AUC  : 0.9995
F1      : 0.9988
Recall  : 1.0000
─────────────────
FINAL   : 0.9997
